## Sentence Transformer (SBERT) + classificadores

In [22]:

import pandas as pd
import numpy as np
import os
import joblib

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Divisão dos dados
from sklearn.model_selection import train_test_split

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

# GridSearch
from sklearn.model_selection import GridSearchCV

# Métricas
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Sentence Transformer
from sentence_transformers import SentenceTransformer

from sklearn.preprocessing import StandardScaler

### 1. Carregamento dos dados de treino

In [56]:
# df_train = pd.read_csv("train.csv")
# df_train = pd.read_csv("train_noDuplicates.csv")
df_train = pd.read_csv("train_noSameClassDuplicates.csv")

# print(df_train.shape)
# print(df_train.columns)
# print(df_train.head())
# print(df_train.isnull().sum())

### 2. Definição das colunas de texto e categoria

In [57]:
X = df_train["resp_text"]
y = df_train["clarity"]

# print("X:", X.shape)
# print("y:", y.shape)
# print("\nDistribuição das classes:")
# print(y.value_counts())
# print("\nProporção das classes:")
# print(y.value_counts(normalize=True))

### 3. Divisão do conjunto em treino e validação

In [58]:
# Separamos  para garantir que o modelo de representação não tenha acesso
# aos dados que vamos usar para avaliar o classificador

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y, # Preserva a distribuição das classes na divisão
    random_state=100
)

print("Treino:", X_train.shape)
print("Validação:", X_val.shape)

print("\nClasses no treino:")
print(y_train.value_counts())

print("\nClasses na validação:")
print(y_val.value_counts())

Treino: (15050,)
Validação: (3763,)

Classes no treino:
clarity
c5      5278
c234    5170
c1      4602
Name: count, dtype: int64

Classes na validação:
clarity
c5      1320
c234    1292
c1      1151
Name: count, dtype: int64


### 4. Carregamento do modelo

In [26]:
model_sbert = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Max seq length:", model_sbert.max_seq_length)
print("Dimensão do embedding:", model_sbert.get_sentence_embedding_dimension())

# model_sbert.max_seq_length = 512
# print("Max seq length:", model_sbert.max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Max seq length: 128
Dimensão do embedding: 384


/tmp/ipykernel_466/857964632.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimensão do embedding:", model_sbert.get_sentence_embedding_dimension())


In [27]:
texto_exemplo = """ Prezado(a) Senhor(a),   Esclarecemos que o Serviço de Informação ao Cidadão (SIC) não trata de sugestão, elogio, solicitação, reclamação e denúncia relacionados a objetos postados no Brasil ou no exterior.   Cumpre-nos esclarecer que a Comissão Mista de Reavaliação de Informações (CMRI), por meio da Súmula nº 01/2015, definiu:   "Caso exista canal ou procedimento específico efetivo para obtenção da informação solicitada, o órgão ou a entidade deve orientar o interessado a buscar a informação por intermédio desse canal ou procedimento, indicando os prazos e as condições para sua utilização, sendo o pedido considerado atendido."   Assim, especificamente sobre o seu pedido de informação, ressaltamos que foi identificado manifestação aberta na Central de Atendimento dos Correios (CAC), e, nesse sentido, ressaltamos que as informações sobre a sua remessa postal serão prestadas a V.S.ª por meio daquele canal de atendimento ao cliente.   Pelo exposto, seguem canais apropriados para tratamento de demandas dessa natureza:   Fale Conosco  registro via internet de reclamações, críticas, sugestões, dúvidas e elogios - http://www2.correios.com.br/sistemas/falecomoscorreios.   Central de Atendimento dos Correios Segunda à sexta-feira: das 08h às 20h. Sábado: 08h às 14h. Domingos e feriados: não há atendimento.   Informações sobre Produtos e Serviços: 3003 0100 (Capitais e regiões metropolitanas), 0800 725 7282 (demais localidades), 0800 725 0898 (exclusivo para portadores de deficiência auditiva), 3003 1383 (Informações Banco Postal).   Reclamações, críticas, sugestões, dúvidas, elogios e denúncias: 0800 725 0100.     Caso a resposta não seja satisfatória, a Ouvidoria da Empresa poderá ser acionada, bastando acessar http://www.correios.com.br/ouvidoria e preencher o formulário correspondente. Ao registrar sua manifestação é preciso informar o protocolo fornecido pelo(a) atendente da Central de Atendimento dos Correios (CAC) ou pelo sistema da internet.     Atenciosamente,   Serviço de Informação ao Cidadão Empresa Brasileira de Correios e Telégrafos   Conforme previsto no art. 21 do Decreto nº 7.724/2012, que regulamenta Lei nº 12.527/2011, eventuais recursos interpostos pelo cidadão no prazo de até 10 dias  contados da ciência da decisão  serão dirigidos ao Vice-Presidente de Operações. """

tokens = model_sbert.tokenizer.encode(
    texto_exemplo,
    add_special_tokens=True,
    truncation=False
)

print("Quantidade total de tokens:", len(tokens))
print("Limite do SBERT:", model_sbert.max_seq_length)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (562 > 128). Running this sequence through the model will result in indexing errors


Quantidade total de tokens: 562
Limite do SBERT: 128


In [28]:
# Testa criar o embedding para a primeira instância dos dados
texto_exemplo = X_train.iloc[0]
print(texto_exemplo)

embedding_exemplo = model_sbert.encode(texto_exemplo)
print(type(embedding_exemplo))
print(embedding_exemplo.shape) # Vetor de 384 números (definido pela arquitetura do modelo)
print(embedding_exemplo[:10])

 Prezada Senhora,  Esclarecemos que conforme preceito da Lei 12.527 de 18 de novembro de 2011-Lei de Acesso à Informação- LAI, o Serviço de Informação ao Cidadão- SIC não é instrumento hábil para o recebimento e processamento de reclamações, elogios, denúncias ou resolução de problemas, e-sic é destinado apenas para acesso a informações de interesse público, sendo vedada para este canal, informações de cunho particular.  Face ao exposto, a demanda de n. 23480023370201711 não poderá ser atendida por meio deste canal. Por oportuno, orientamos que para informações particulares, utilize a Central de Atendimento ao Cidadão do FNDE através do telefone 0800-616161 ou o link FALE CONOSCO, localizado no Portal do FNDE:  www.fnde.gov.br, a fim de que sua demanda seja prontamente atendida pela equipe técnica correspondente.  Passo a passo para abertura de demandas:  1 - Acessar o Portal FNDE (www.fnde.gov.br) na barra de endereço do seu navegador, clicar no link Fale Conosco, 2 - Inserir na tel

### 5. Criação dos embeddings de fato

In [59]:
# Cria os embeddings para todos os dados de treino (~15min)
X_train_embeddings = model_sbert.encode(
    X_train.tolist(),
    show_progress_bar=True
)
print("Treino:", X_train_embeddings.shape)

# X_train_embeddings_chunked = np.array([
#     encode_text_with_chunks(
#         text,
#         model_sbert
#     )
#     for text in tqdm(
#         X_train,
#         desc="Embeddings de treino"
#     )
# ])

# print("Treino:", X_train_embeddings_chunked.shape)

Batches:   0%|          | 0/471 [00:00<?, ?it/s]

Treino: (15050, 384)


In [60]:
# Cria embeddings para todos os dados de validação (~5min)
X_val_embeddings = model_sbert.encode(
    X_val.tolist(),
    show_progress_bar=True
)
print("Validação:", X_val_embeddings.shape)

# X_val_embeddings_chunked = np.array([
#     encode_text_with_chunks(
#         text,
#         model_sbert
#     )
#     for text in tqdm(
#         X_val,
#         desc="Embeddings de validação"
#     )
# ])

# print("Validação:", X_val_embeddings_chunked.shape)

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Validação: (3763, 384)


### 6. Criação do modelo SBERT + Regressão Logística

In [31]:
# pipeline_sbert_lr = Pipeline(
#     steps=[
#         ("clf", LogisticRegression(
#             class_weight='balanced',
#             max_iter=1000,
#             random_state=100
#         ))
#     ]
# )

### 6.1 Treinamento

In [32]:
# Aqui ele aprende a relação entre os 384 valores dos embeddings e as três classes
# pipeline_sbert_lr.fit(
#     X_train_embeddings,
#     y_train
# )

### 6.2 Fazer as previsões no conjunto de validação

In [33]:
# y_pred = pipeline_sbert_lr.predict(X_val_embeddings)

# accuracy = accuracy_score(y_val, y_pred)
# f1_macro = f1_score(y_val, y_pred, average="macro")

# print(f"Accuracy: {accuracy:.4f}")
# print(f"Macro-F1: {f1_macro:.4f}")

# print("\nClassification Report:")
# print(classification_report(y_val, y_pred))

### 6.3 Salvando pipeline - Regressão Logística

In [34]:
# joblib.dump(
#     pipeline_sbert_lr,
#     "pipelines/pipelineSBERT_LR.pkl"
# )

### 7. Criação do modelo SBERT + SVM

In [35]:
pipeline_sbert_svm = Pipeline(
    steps=[
        ("clf", SVC(
            class_weight="balanced"
        ))
    ]
)

# pipeline_sbert_chunked_svm = Pipeline(
#     steps=[
#         ("scaler", StandardScaler()),
#         ("clf", SVC(
#             class_weight="balanced"
#         ))
#     ]
# )

### 7.1 Treinamento

In [36]:
pipeline_sbert_svm.fit(
    X_train_embeddings,
    y_train
)

# pipeline_sbert_chunked_svm.fit(
#     X_train_embeddings_chunked,
#     y_train
# )

Pipeline(steps=[('clf', SVC(class_weight='balanced'))])

### 7.2 Fazer as previsões no conjunto de validação

In [37]:
y_pred_svm = pipeline_sbert_svm.predict(
    X_val_embeddings
)

accuracy_svm = accuracy_score(y_val, y_pred_svm)
f1_macro_svm = f1_score(y_val, y_pred_svm, average="macro")

print(f"Accuracy: {accuracy_svm:.4f}")
print(f"Macro-F1: {f1_macro_svm:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred_svm))

# y_pred_chunked_svm = pipeline_sbert_chunked_svm.predict(
#     X_val_embeddings_chunked
# )

# accuracy_chunked_svm = accuracy_score(
#     y_val,
#     y_pred_chunked_svm
# )

# f1_macro_chunked_svm = f1_score(
#     y_val,
#     y_pred_chunked_svm,
#     average="macro"
# )

# print(f"Accuracy: {accuracy_chunked_svm:.4f}")
# print(f"Macro-F1: {f1_macro_chunked_svm:.4f}")

# print("\nClassification Report:")
# print(
#     classification_report(
#         y_val,
#         y_pred_chunked_svm
#     )
# )

Accuracy: 0.4362
Macro-F1: 0.4341

Classification Report:
              precision    recall  f1-score   support

          c1       0.45      0.52      0.48      1269
        c234       0.38      0.33      0.35      1371
          c5       0.46      0.47      0.47      1379

    accuracy                           0.44      4019
   macro avg       0.43      0.44      0.43      4019
weighted avg       0.43      0.44      0.43      4019



### 7.3 Salva pipeline - SVM

In [38]:
# joblib.dump(
#     pipeline_sbert_svm,
#     "pipelines/pipelineSBERT_SVM.pkl"
# )

### 8. Criação do modelo SBERT + MLP

In [39]:
# pipeline_sbert_mlp = Pipeline(
#     steps=[
#         ("clf", MLPClassifier(
#             hidden_layer_sizes=(128,),
#             max_iter=300,
#             random_state=100
#         ))
#     ]
# )

### 8.1 Treinamento

In [40]:
# pipeline_sbert_mlp.fit(
#     X_train_embeddings,
#     y_train
# )

### 8.2 Fazer as previsões no conjunto de validação

In [41]:
# y_pred_mlp = pipeline_sbert_mlp.predict(
#     X_val_embeddings
# )

# accuracy_mlp = accuracy_score(y_val, y_pred_mlp)
# f1_macro_mlp = f1_score(y_val, y_pred_mlp, average="macro")

# print(f"Accuracy: {accuracy_mlp:.4f}")
# print(f"Macro-F1: {f1_macro_mlp:.4f}")

# print("\nClassification Report:")
# print(classification_report(y_val, y_pred_mlp))

### 9. Fazendo GridSearch para modelo SVM
"Melhor" modelo dos três

In [42]:
from sklearn.model_selection import StratifiedKFold

# Parâmetros a testar
# param_grid_svm = {
#     "clf__C": [0.01, 0.1, 1, 10, 100],
#     "clf__kernel": ["linear", "rbf"],
#     "clf__gamma": ["scale", "auto"]
# }

# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# grid_svm = GridSearchCV(
#     estimator=pipeline_sbert_svm, # nosso pipeline SBERT + SVM
#     param_grid=param_grid_svm, # combinações que queremos testar
#     scoring="accuracy",
#     cv=cv, # cada combinação será avaliada em 5 divisões do conjunto de treino
#     n_jobs=-1, # usa os processadores disponíveis para paralelizar
#     return_train_score=True # guarda também o desempenho no treino, para investigar overfitting
# )

### 9.1 Treinamento do GridSearch

In [43]:
# Faz o treinamento usando 30% o conjunto de treino (exploração dos melhores parâmetros)

# X_train_sample, _, y_train_sample, _ = train_test_split(
#     X_train_embeddings,
#     y_train,
#     train_size=0.30,
#     stratify=y_train,
#     random_state=100
# )

# print(X_train_sample.shape)
# print(y_train_sample.value_counts())

# grid_svm.fit( # ~15min
#     X_train_sample,
#     y_train_sample
# )

# # Ver qual a melhor configuração
# print("Melhores parâmetros:")
# print(grid_svm.best_params_)

# print("\nMelhor Accuracy média na validação cruzada:")
# print(f"{grid_svm.best_score_:.4f}")

(4821, 384)
clarity
c5      1654
c234    1644
c1      1523
Name: count, dtype: int64


KeyboardInterrupt: 

### 9.2 Avaliar na validação

In [ ]:
# Testa a performance da melhor configuração escolhida no conjunto de validação
# y_pred_svm_grid = grid_svm.predict(
#     X_val_embeddings
# )

# accuracy_svm_grid = accuracy_score(y_val, y_pred_svm_grid)

# f1_macro_svm_grid = f1_score(
#     y_val,
#     y_pred_svm_grid,
#     average="macro"
# )

# print(f"Accuracy: {accuracy_svm_grid:.4f}")
# print(f"Macro-F1: {f1_macro_svm_grid:.4f}")

# print("\nClassification Report:")
# print(classification_report(y_val, y_pred_svm_grid))

### **10. Modelo com as melhores configurações**
Fazendo validação cruzada e variando a seed

In [61]:
pipeline_svm_best = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        C=1,
        kernel="rbf",
        gamma="scale",
        class_weight="balanced"
    ))
])

In [62]:
from sklearn.model_selection import StratifiedKFold, cross_validate

seeds = [42, 100, 123]

for seed in seeds:

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    cv_results = cross_validate(
        pipeline_svm_best,
        X_train_embeddings,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro"
        },
        n_jobs=-1
    )

    print(f"\nSeed: {seed}")
    print(f"Accuracy: {cv_results['test_accuracy'].mean():.4f} "
          f"+/- {cv_results['test_accuracy'].std():.4f}")
    print(f"Macro-F1: {cv_results['test_f1_macro'].mean():.4f} "
          f"+/- {cv_results['test_f1_macro'].std():.4f}")

KeyboardInterrupt: 

In [63]:
# Treina em TODO o conjunto de treinamento
pipeline_svm_best.fit(X_train_embeddings, y_train)

# Avaliação no conjunto externo de validação
y_pred = pipeline_svm_best.predict(X_val_embeddings)

accuracy = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average="macro")

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Macro-F1: {f1:.4f}")

Validation Accuracy: 0.4401
Validation Macro-F1: 0.4382
